# 21 · Cierre de la comparación de recuperación

Reutiliza `evaluacion_recuperacion_v1.json` y `densa_resultados_v1.json`.
La recomendación de trabajo maximiza el **límite inferior de nDCG@10** en comparación,
promediando ES/EN por necesidad. La regla se propone **después del análisis**;
no es una regla preregistrada ni demuestra superioridad estadística.

No ejecuta modelos, no modifica juicios y no recalcula embeddings.
Salida única: `seleccion_recuperador_v1.json`. La recomendación queda **provisional**.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

ARCHIVOS = {"evaluacion": "evaluacion_recuperacion_v1.json",
            "densa": "densa_resultados_v1.json"}
SALIDA = "seleccion_recuperador_v1.json"
VERSION = "CIERRE_RECUPERACION_V1"
TOL = 1e-12

def exigir(condicion, mensaje):
    if not condicion:
        raise ValueError(mensaje)

def firma(objeto):
    texto = json.dumps(objeto, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

carpetas = [p / s for p in (Path.cwd(), *Path.cwd().parents)
            for s in (".", "07_RAG/Pruebas_RAG", "Pruebas_RAG")]
CARPETA = next((p for p in carpetas if (p / ARCHIVOS["evaluacion"]).is_file()), None)
if CARPETA is None:
    raise FileNotFoundError("Falta evaluacion_recuperacion_v1.json en 07_RAG/Pruebas_RAG/.")
entradas = {}
for clave, nombre in ARCHIVOS.items():
    ruta = CARPETA / nombre
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta este archivo exacto: {ruta.resolve()}")
    entradas[clave] = ruta.read_bytes()
huellas = {k: hashlib.sha256(v).hexdigest() for k, v in entradas.items()}
E, D = [json.loads(entradas[k]) for k in ARCHIVOS]
R = E["resultados"]
exigir(firma(R) == E["firma_resultados"], "La firma de la evaluación no coincide.")
exigir(huellas["densa"] == R["fuentes"]["densa"]["sha256"],
       "La recuperación densa no corresponde a esta evaluación. Conserva las versiones compatibles.")
exigir(R["validacion"]["pendientes"] == 0, "Quedan casos sin revisar en la evaluación.")
exigir(R["validacion"]["sistemas"] == 23 and len(D["sistemas"]) == 11,
       "Se esperan 23 sistemas y 11 codificadores densos.")
print("Evaluación completa y recuperación densa compatibles.")
print("Juicios humanos con grado:", R["validacion"]["juicios_humanos_con_grado"])


Evaluación completa y recuperación densa compatibles.
Juicios humanos con grado: 0


## 1. Calidad, recursos y recomendación de trabajo

Los rangos varían únicamente los grados desconocidos; no son intervalos de confianza.
Las latencias de los híbridos corresponden solo a fusión: no se equiparan al tiempo total denso.


In [2]:
resumen = pd.DataFrame(R["resumen"])
recursos = pd.DataFrame(R["recursos"])
comparacion = resumen.query("division == 'comparacion' and idioma == 'es_en' and k == 10").copy()
comparacion = comparacion.merge(recursos[["sistema", "familia", "latencia_mediana_ms_comparacion",
    "alcance_latencia", "pico_pytorch_asignado_bytes"]], on="sistema", validate="one_to_one")
comparacion = comparacion.sort_values(["ndcg_juzgados_inf", "sistema"], ascending=[False, True])
escenarios = pd.DataFrame([e["puntajes"] for e in R["escenarios_comparacion_ndcg10"]])
exigir(set(escenarios) == set(comparacion["sistema"]), "Sistemas inconsistentes.")
for fila in comparacion.to_dict("records"):
    v = escenarios[fila["sistema"]]
    exigir(np.isclose(v.min(), fila["ndcg_juzgados_inf"], atol=TOL, rtol=0)
           and np.isclose(v.max(), fila["ndcg_juzgados_sup"], atol=TOL, rtol=0),
           "Los escenarios no reproducen el rango de nDCG@10.")

mejores = np.isclose(escenarios.to_numpy(), escenarios.max(axis=1).to_numpy()[:, None], atol=TOL, rtol=0)
finalistas = sorted(escenarios.columns[mejores.any(axis=0)].tolist())
limite = comparacion["ndcg_juzgados_inf"].max()
candidatos = comparacion.loc[np.isclose(comparacion["ndcg_juzgados_inf"], limite, atol=TOL, rtol=0), "sistema"]
exigir(len(candidatos) == 1, "Empate en el criterio conservador: no se elige por velocidad automáticamente.")
RECOMENDADO = candidatos.iloc[0]
exigir(RECOMENDADO in D["sistemas"], "La recomendación no es densa; este cierre debe adaptarse.")
print("Recomendación provisional:", RECOMENDADO)
print("Primeros posibles en los escenarios actuales:", finalistas)
display(comparacion[["sistema", "familia", "ndcg_juzgados_inf", "ndcg_juzgados_sup",
    "precision_inf", "precision_sup", "recall_juzgados_inf", "recall_juzgados_sup"]].round(4).reset_index(drop=True))

densos = comparacion.query("familia == 'denso'").copy()
densos["pico_PyTorch_GiB"] = densos["pico_pytorch_asignado_bytes"] / 2**30
display(densos[["sistema", "latencia_mediana_ms_comparacion", "pico_PyTorch_GiB"]].round(3).reset_index(drop=True))
idiomas = resumen.query("division == 'comparacion' and k == 10 and idioma != 'es_en'")
ids_muestra = ["bm25", RECOMENDADO, "rrf_bm25_" + RECOMENDADO]
display(idiomas[idiomas["sistema"].isin(ids_muestra)][["sistema", "idioma", "ndcg_juzgados_inf", "ndcg_juzgados_sup"]].round(4))


Recomendación provisional: qwen3_4b
Primeros posibles en los escenarios actuales: ['arctic_l_v2', 'nemotron3_1b', 'pplx_4b_bf16', 'qwen3_4b']


,sistema,familia,ndcg_juzgados_inf,ndcg_juzgados_sup,precision_inf,precision_sup,recall_juzgados_inf,recall_juzgados_sup
0,qwen3_4b,denso,0.9438,0.9742,0.2250,0.2500,0.9196,0.9375
1,arctic_l_v2,denso,0.9358,0.9649,0.2188,0.2438,0.9107,0.9297
2,harrier_0_6b,denso,0.9339,0.9672,0.2188,0.2375,0.9062,0.9286
3,nemotron3_1b,denso,0.9327,0.9762,0.2313,0.2438,0.9219,0.9531
4,pplx_4b_bf16,denso,0.9325,0.9764,0.2313,0.2438,0.9219,0.9531
5,qwen3_vl_2b_text,denso,0.9302,0.9745,0.2250,0.2375,0.9062,0.9375
6,pplx_0_6b,denso,0.9294,0.9636,0.2125,0.2313,0.8984,0.9196
7,embeddinggemma2_text,denso,0.9224,0.9632,0.2125,0.2250,0.8984,0.9263
8,qwen3_0_6b,denso,0.9204,0.9642,0.2188,0.2313,0.8906,0.9219
9,granite_311m_r2,denso,0.9166,0.9588,0.2125,0.2188,0.8889,0.9167


,sistema,latencia_mediana_ms_comparacion,pico_PyTorch_GiB
0,qwen3_4b,61.814,7.561
1,arctic_l_v2,19.975,2.126
2,harrier_0_6b,46.083,1.122
3,nemotron3_1b,24.134,2.137
4,pplx_4b_bf16,60.258,7.557
5,qwen3_vl_2b_text,48.253,3.977
6,pplx_0_6b,44.658,2.232
7,embeddinggemma2_text,56.754,0.518
8,qwen3_0_6b,45.889,1.123
9,granite_311m_r2,19.702,0.598


,sistema,idioma,ndcg_juzgados_inf,ndcg_juzgados_sup
10,bm25,es,0.0000,0.0000
13,bm25,en,0.8349,0.8751
172,qwen3_4b,es,0.9434,0.9745
175,qwen3_4b,en,0.9431,0.9740
370,rrf_bm25_qwen3_4b,es,0.1808,0.1808
373,rrf_bm25_qwen3_4b,en,0.8797,0.9064


## 2. Sensibilidad de la decisión

Se compara cada sistema bajo la misma asignación de los grados desconocidos.
El diagnóstico de omitir una necesidad es posterior al análisis: **no cambia** el conjunto principal,
no selecciona qué temas conservar y no es una prueba de significancia.


In [3]:
pareadas = []
for sistema in escenarios:
    delta = escenarios[RECOMENDADO] - escenarios[sistema]
    pareadas.append({"frente_a": sistema, "delta_ndcg10_min": float(delta.min()),
                    "delta_ndcg10_max": float(delta.max()),
                    "recomendado_superior_en_todos_los_escenarios": bool((delta > TOL).all())})

m = pd.DataFrame(R["metricas_por_consulta"])
m = m.query("division == 'comparacion' and k == 10")
por_necesidad = m.groupby(["sistema", "necesidad_id"])[["ndcg_juzgados_inf", "ndcg_juzgados_sup"]].mean()
ns = sorted(m["necesidad_id"].unique())
ns_nulas = sorted({c["necesidad_id"] for c in R["casos_informacion_insuficiente"] if c["necesidad_id"] in ns})
exigir(len(ns_nulas) == 1, "Este diagnóstico requiere una sola necesidad de comparación con grados null.")
conocidas = por_necesidad.reset_index().query("necesidad_id not in @ns_nulas")
exigir(np.allclose(conocidas.ndcg_juzgados_inf, conocidas.ndcg_juzgados_sup, atol=TOL, rtol=0),
       "Hay incertidumbre adicional fuera de la necesidad identificada.")
fijos = conocidas.pivot(index="sistema", columns="necesidad_id", values="ndcg_juzgados_inf")
empates_densos = [n for n in fijos if np.ptp(fijos.loc[list(D["sistemas"]), n]) <= TOL]
sensibilidad_necesidades = []
for n in ns:
    valores = (pd.DataFrame([fijos.mean(axis=1)]).reindex(columns=escenarios.columns)
               if n in ns_nulas else (len(ns) * escenarios - fijos[n]) / (len(ns)-1))
    mejor = np.isclose(valores.to_numpy(), valores.max(axis=1).to_numpy()[:, None], atol=TOL, rtol=0)
    sensibilidad_necesidades.append({"necesidad_omitida_solo_en_diagnostico": n,
        "primeros_posibles": sorted(valores.columns[mejor.any(axis=0)].tolist())})
print("Necesidades sin diferencias de nDCG@10 entre los 11 densos:", empates_densos)
display(pd.DataFrame(sensibilidad_necesidades))
print("No se elimina ninguna necesidad ni se cambian los juicios.")
print("Los casos sin grado de comparación tienen prioridad para revisión:",
      [c for c in R["casos_informacion_insuficiente"] if c["necesidad_id"] in ns])


Necesidades sin diferencias de nDCG@10 entre los 11 densos: ['N06', 'N08', 'N09', 'N10', 'N12']


,necesidad_omitida_solo_en_diagnostico,primeros_posibles
0,N05,"[arctic_l_v2, qwen3_4b, qwen3_vl_2b_text]"
1,N06,"[arctic_l_v2, nemotron3_1b, pplx_4b_bf16, qwen..."
2,N07,[nemotron3_1b]
3,N08,"[arctic_l_v2, nemotron3_1b, pplx_4b_bf16, qwen..."
4,N09,"[arctic_l_v2, nemotron3_1b, pplx_4b_bf16, qwen..."
5,N10,"[arctic_l_v2, nemotron3_1b, pplx_4b_bf16, qwen..."
6,N11,"[arctic_l_v2, nemotron3_1b, pplx_4b_bf16, qwen..."
7,N12,"[arctic_l_v2, nemotron3_1b, pplx_4b_bf16, qwen..."


No se elimina ninguna necesidad ni se cambian los juicios.
Los casos sin grado de comparación tienen prioridad para revisión: [{'necesidad_id': 'N07', 'article_id': '255'}, {'necesidad_id': 'N07', 'article_id': '57'}, {'necesidad_id': 'N07', 'article_id': '5'}]


## 3. Guardar el cierre

El archivo conserva la configuración exacta del codificador y las huellas de sus artefactos.
`k=10` es el corte de evaluación; **no determina** cuántos artículos recibirá Gemma.
La recomendación no equivale a una decisión aprobada por el usuario ni a validación humana.


In [4]:
def registros(tabla):
    return tabla.astype(object).where(pd.notna(tabla), None).to_dict("records")

sistema = D["sistemas"][RECOMENDADO]
limitaciones = [
    "Evaluación asistida exploratoria; no cuenta con validación humana independiente.",
    "El criterio conservador se definió después del análisis, no fue preregistrado.",
    "Los extremos solo cubren los null; no cubren errores en otros grados ni relevantes no juzgados.",
    "Solo ocho necesidades de comparación; ES/EN son formulaciones pareadas, no 16 tareas independientes.",
    "Las consultas proceden de artículos orientadores. No prueban generalización a necesidades nuevas.",
    "Cinco necesidades no distinguen a los densos con nDCG@10; la selección depende de los otros temas.",
    "La memoria es pico asignado por PyTorch, no memoria total. No se ha medido coexistencia con Gemma.",
    "RRF se descarta solo en la configuración probada; no se concluye que toda recuperación híbrida sea inferior.",
    "No se modifica el protocolo de consulta ni la precisión del codificador seleccionado.",
    "No se valida la calidad de justificaciones de coautoría ni los factores del predictor de enlaces.",
]
cuerpo = {
    "version_esquema": VERSION,
    "fuentes": {k: {"archivo": ARCHIVOS[k], "sha256": huellas[k]} for k in ARCHIVOS},
    "firma_evaluacion": E["firma_resultados"], "firma_conjunto": R["firma_conjunto"],
    "estado": "recomendacion_operativa_provisional_del_asistente",
    "aprobacion_usuario_registrada": False, "seleccion_definitiva": False,
    "validacion_humana_independiente": False,
    "recomendacion": {"sistema": RECOMENDADO, "familia": "denso",
        "modelo": sistema["condicion"]["modelo"],
        "regla": "Mayor límite inferior de nDCG_juzgados@10, comparación, promedio ES/EN por necesidad.",
        "momento_definicion_regla": "posterior_al_analisis_de_resultados",
        "calidad_y_eficiencia_combinadas_en_un_puntaje": False,
        "k_evaluacion": 10, "k_contexto_gemma": None,
        "umbral_similitud_calibrado": None},
    "firma_condicion_documentada": sistema["firma_condicion"],
    "configuracion_evaluada": sistema["condicion"],
    "modelo_runtime_evaluado": sistema["modelo_runtime"],
    "artefactos_documentales_existentes": {"carpeta_en_repositorio": "07_RAG", **sistema["fuentes"]},
    "resultado_recomendado": registros(comparacion.query("sistema == @RECOMENDADO"))[0],
    "recursos_recomendado": next(r for r in R["recursos"] if r["sistema"] == RECOMENDADO),
    "comparacion_23_sistemas": registros(comparacion),
    "primeros_posibles_en_escenarios_actuales": finalistas,
    "diferencias_pareadas_frente_a_recomendado": pareadas,
    "sensibilidad_al_omitir_una_necesidad": sensibilidad_necesidades,
    "necesidades_sin_diferencias_entre_densos": empates_densos,
    "casos_sin_grado": R["casos_informacion_insuficiente"],
    "validacion_recomendada": [
        "Priorizar los tres casos ambiguos de N07 sin mirar rankings al juzgar pertinencia.",
        "Contrastar los juicios positivos y una muestra de ceros con un evaluador humano independiente.",
        "Conservar revisiones en otra versión, registrar su autor y recalcular métricas sin nuevas inferencias.",
        "Evaluar después necesidades nuevas vinculadas a pares de autores, sin usar esta comparación como prueba final de coautoría."
    ],
    "limitaciones": limitaciones,
    "siguiente_actividad": "Preparar el piloto de recuperación para pares de autores, conservando article_id y separando evidencia recuperada de factores del predictor.",
}
resultado = {"firma_seleccion": firma(cuerpo), "seleccion": cuerpo}
for k, nombre in ARCHIVOS.items():
    exigir(hashlib.sha256((CARPETA/nombre).read_bytes()).hexdigest() == huellas[k], "Una entrada cambió.")
def equivalentes(x, y):
    if isinstance(x, dict) and isinstance(y, dict):
        return x.keys() == y.keys() and all(equivalentes(x[k], y[k]) for k in x)
    if isinstance(x, list) and isinstance(y, list):
        return len(x) == len(y) and all(equivalentes(a, b) for a, b in zip(x, y))
    if type(x) in (int, float) and type(y) in (int, float):
        return bool(np.isclose(x, y, atol=TOL, rtol=0))
    return x == y

destino = CARPETA / SALIDA
if destino.exists():
    anterior = json.loads(destino.read_text(encoding="utf-8"))
    exigir(firma(anterior["seleccion"]) == anterior["firma_seleccion"], "La salida existente fue alterada.")
    exigir(equivalentes(anterior["seleccion"], cuerpo),
           "Existe otro cierre. Consérvalo y cambia SALIDA para guardar una nueva versión.")
    print("Cierre equivalente: se conserva el archivo existente.")
else:
    temporal = destino.with_suffix(".tmp")
    temporal.write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporal.replace(destino)
print("Archivo:", destino.resolve())
print("Recomendación provisional:", sistema["condicion"]["modelo"], "— recuperación densa")
print("Sin cambios en fuentes, anotaciones, rankings o embeddings.")


Archivo: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\seleccion_recuperador_v1.json
Recomendación provisional: Qwen/Qwen3-Embedding-4B — recuperación densa
Sin cambios en fuentes, anotaciones, rankings o embeddings.
